## **NBA Prospect Evaluator**
##### *Predicting NBA success based on NCAA statistics*

In [1]:
print("Jared Mccain is my favorite nba player of all time")

Jared Mccain is my favorite nba player of all time


In [2]:
# Test Environment
print("Jared McCain")

Jared McCain


# Data Collection

#### Use API to get college basketball player data *(getcollegedata.py)*

In [4]:
from getcollegedata import pull_all_seasons
import os
import pandas as pd

if os.path.exists("data/raw/player_stats_2015_2025.csv"):
    df = pd.read_csv(
        "data/raw/player_stats_2015_2025.csv"
    )
else:
    df = pull_all_seasons(2015, 2025)
    df.to_csv(
        "data/raw/player_stats_2015_2025.csv",
        index=False
    )

In [48]:
college_player_stats = pd.read_csv("data/raw/player_stats_2015_2025.csv")
college_player_stats.columns
college_player_stats.head()

,season,seasonLabel,teamId,team,conference,athleteId,athleteSourceId,name,position,games,...,freeThrows.pct,freeThrows.attempted,freeThrows.made,rebounds.total,rebounds.defensive,rebounds.offensive,winShares.totalPer40,winShares.total,winShares.defensive,winShares.offensive
0,2015,20142015,1,Abilene Christian,Southland,47237,3152345,K.J. Maura,G,17,...,100.0,5,5,24,17,7,0.041,0.2,0.1,0.1
1,2015,20142015,1,Abilene Christian,Southland,47386,3152349,Drake Green,G,28,...,56.3,16,9,22,18,4,-0.049,-0.3,0.1,-0.4
2,2015,20142015,1,Abilene Christian,Southland,47387,3152347,Isaiah Tripp,G,29,...,60.0,25,15,33,20,13,0.057,0.3,0.1,0.2
3,2015,20142015,1,Abilene Christian,Southland,52198,67296,Christian Albright,F,20,...,64.7,17,11,65,44,21,0.015,0.1,0.3,-0.2
4,2015,20142015,1,Abilene Christian,Southland,56233,3155902,Duran Porter,F,31,...,50.0,40,20,71,42,29,0.010,0.1,0.2,-0.1


In [49]:
# Add additional columns
import numpy as np

# Per-game stats
college_player_stats["MPG"] = np.where(
    college_player_stats["games"] > 0,
    college_player_stats["minutes"] / college_player_stats["games"],
    np.nan
)

college_player_stats["PPG"] = np.where(
    college_player_stats["games"] > 0,
    college_player_stats["points"] / college_player_stats["games"],
    np.nan
)

college_player_stats["APG"] = np.where(
    college_player_stats["games"] > 0,
    college_player_stats["assists"] / college_player_stats["games"],
    np.nan
)

college_player_stats["SPG"] = np.where(
    college_player_stats["games"] > 0,
    college_player_stats["steals"] / college_player_stats["games"],
    np.nan
)

college_player_stats["BPG"] = np.where(
    college_player_stats["games"] > 0,
    college_player_stats["blocks"] / college_player_stats["games"],
    np.nan
)

college_player_stats["TPG"] = np.where(
    college_player_stats["games"] > 0,
    college_player_stats["turnovers"] / college_player_stats["games"],
    np.nan
)

college_player_stats["FPG"] = np.where(
    college_player_stats["games"] > 0,
    college_player_stats["fouls"] / college_player_stats["games"],
    np.nan
)

# Per-40 stats
college_player_stats["PTS_PER_40"] = np.where(
    college_player_stats["minutes"] > 0,
    college_player_stats["points"] / college_player_stats["minutes"] * 40,
    np.nan
)

college_player_stats["AST_PER_40"] = np.where(
    college_player_stats["minutes"] > 0,
    college_player_stats["assists"] / college_player_stats["minutes"] * 40,
    np.nan
)

college_player_stats["STL_PER_40"] = np.where(
    college_player_stats["minutes"] > 0,
    college_player_stats["steals"] / college_player_stats["minutes"] * 40,
    np.nan
)

college_player_stats["BLK_PER_40"] = np.where(
    college_player_stats["minutes"] > 0,
    college_player_stats["blocks"] / college_player_stats["minutes"] * 40,
    np.nan
)

college_player_stats["TOV_PER_40"] = np.where(
    college_player_stats["minutes"] > 0,
    college_player_stats["turnovers"] / college_player_stats["minutes"] * 40,
    np.nan
)

college_player_stats["FOULS_PER_40"] = np.where(
    college_player_stats["minutes"] > 0,
    college_player_stats["fouls"] / college_player_stats["minutes"] * 40,
    np.nan
)

# Additional prospect metrics
college_player_stats["AST_TO_TOV"] = np.where(
    college_player_stats["turnovers"] > 0,
    college_player_stats["assists"] / college_player_stats["turnovers"],
    np.nan
)

college_player_stats["STOCKS_PER_40"] = (
    college_player_stats["STL_PER_40"] +
    college_player_stats["BLK_PER_40"]
)

# Round all created columns
new_cols = [
    "MPG", "PPG", "APG", "SPG", "BPG", "TPG", "FPG",
    "PTS_PER_40", "AST_PER_40", "STL_PER_40",
    "BLK_PER_40", "TOV_PER_40", "FOULS_PER_40",
    "AST_TO_TOV", "STOCKS_PER_40"
]

college_player_stats[new_cols] = (
    college_player_stats[new_cols].round(2)
)

NBA Player Data

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

all_dfs = []


if os.path.exists("data/raw/nba_player_stats_2016_2026.csv"):
    print("File already exists.")
else:
    for year in range(16, 27):  # 16 through 26
        file_path = ROOT / "data" / "raw" / f"nba_player_data_{year}.xlsx"

        print(f"Loading {file_path.name}")

        df = pd.read_excel(file_path)

        # add a season column
        df["season"] = year

        all_dfs.append(df)

    nba_player_stats = pd.concat(all_dfs, ignore_index=True)

    # Save combined dataset
    output_path = ROOT / "data" / "raw" / "nba_player_stats_2016_2026.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    nba_player_stats.to_csv(output_path, index=False)

    print(f"Saved to {output_path}")
    


Loading nba_player_data_16.xlsx
Loading nba_player_data_17.xlsx
Loading nba_player_data_18.xlsx
Loading nba_player_data_19.xlsx
Loading nba_player_data_20.xlsx
Loading nba_player_data_21.xlsx
Loading nba_player_data_22.xlsx
Loading nba_player_data_23.xlsx
Loading nba_player_data_24.xlsx
Loading nba_player_data_25.xlsx
Loading nba_player_data_26.xlsx
Saved to c:\Users\jftho\OneDrive\Documents\NBA_Prospect_Evaluator\data\raw\nba_player_stats_2016_2026.csv


Clean Data

In [33]:
# Load NBA player stats into dataframe
nba_player_stats = pd.read_csv("data/raw/nba_player_stats_2016_2026.csv")
nba_player_stats.head()

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,TRB,AST,STL,BLK,TOV,PF,PTS,Trp-Dbl,Awards,season
0,1,James Harden,26,HOU,SG,82,82,3125,710,1617,...,501,612,139,51,374,229,2376,3,"MVP-9,AS",16
1,2,Stephen Curry,27,GSW,PG,79,79,2700,805,1598,...,430,527,169,15,262,161,2375,2,"MVP-1,AS,NBA1",16
2,3,Kevin Durant,27,OKC,SF,72,72,2578,698,1381,...,589,361,69,85,250,137,2029,1,"MVP-5,AS,NBA2",16
3,4,LeBron James,31,CLE,SF,76,76,2709,737,1416,...,565,514,104,49,249,143,1920,3,"MVP-3,DPOY-11,AS,NBA1",16
4,5,Damian Lillard,25,POR,PG,75,75,2676,618,1474,...,302,512,65,28,242,165,1879,0,"MVP-8,NBA2",16


In [ ]:
# Sort so the row with the most games is first
nba_player_stats = nba_player_stats.sort_values(
    ["Player", "season", "G"],
    ascending=[True, True, False]
)

# Keep only the first row for each player-season
nba_player_stats = nba_player_stats.drop_duplicates(
    subset=["Player", "season"],
    keep="first"
).reset_index(drop=True)

In [ ]:
# Add columns to nba player stats df
import numpy as np

# Per-game stats
nba_player_stats["MPG"] = np.where(
    nba_player_stats["G"] > 0,
    nba_player_stats["MP"] / nba_player_stats["G"],
    np.nan
)

nba_player_stats["PPG"] = np.where(
    nba_player_stats["G"] > 0,
    nba_player_stats["PTS"] / nba_player_stats["G"],
    np.nan
)

nba_player_stats["APG"] = np.where(
    nba_player_stats["G"] > 0,
    nba_player_stats["AST"] / nba_player_stats["G"],
    np.nan
)

nba_player_stats["RPG"] = np.where(
    nba_player_stats["G"] > 0,
    nba_player_stats["TRB"] / nba_player_stats["G"],
    np.nan
)

nba_player_stats["ORPG"] = np.where(
    nba_player_stats["G"] > 0,
    nba_player_stats["ORB"] / nba_player_stats["G"],
    np.nan
)

nba_player_stats["DRPG"] = np.where(
    nba_player_stats["G"] > 0,
    nba_player_stats["DRB"] / nba_player_stats["G"],
    np.nan
)

# Per-40 stats
nba_player_stats["PTS_PER_40"] = np.where(
    nba_player_stats["MP"] > 0,
    nba_player_stats["PTS"] / nba_player_stats["MP"] * 40,
    np.nan
)

nba_player_stats["AST_PER_40"] = np.where(
    nba_player_stats["MP"] > 0,
    nba_player_stats["AST"] / nba_player_stats["MP"] * 40,
    np.nan
)

nba_player_stats["REB_PER_40"] = np.where(
    nba_player_stats["MP"] > 0,
    nba_player_stats["TRB"] / nba_player_stats["MP"] * 40,
    np.nan
)

nba_player_stats["ORB_PER_40"] = np.where(
    nba_player_stats["MP"] > 0,
    nba_player_stats["ORB"] / nba_player_stats["MP"] * 40,
    np.nan
)

nba_player_stats["DRB_PER_40"] = np.where(
    nba_player_stats["MP"] > 0,
    nba_player_stats["DRB"] / nba_player_stats["MP"] * 40,
    np.nan
)

nba_player_stats["STL_PER_40"] = np.where(
    nba_player_stats["MP"] > 0,
    nba_player_stats["STL"] / nba_player_stats["MP"] * 40,
    np.nan
)

nba_player_stats["BLK_PER_40"] = np.where(
    nba_player_stats["MP"] > 0,
    nba_player_stats["BLK"] / nba_player_stats["MP"] * 40,
    np.nan
)

nba_player_stats["TOV_PER_40"] = np.where(
    nba_player_stats["MP"] > 0,
    nba_player_stats["TOV"] / nba_player_stats["MP"] * 40,
    np.nan
)

nba_player_stats["PF_PER_40"] = np.where(
    nba_player_stats["MP"] > 0,
    nba_player_stats["PF"] / nba_player_stats["MP"] * 40,
    np.nan
)

nba_player_stats["AST_TO_TOV"] = np.where(
    nba_player_stats["TOV"] > 0,
    nba_player_stats["AST"] / nba_player_stats["TOV"],
    np.nan
)

nba_player_stats["STOCKS_PER_40"] = (
    nba_player_stats["STL_PER_40"] +
    nba_player_stats["BLK_PER_40"]
)

# Round all newly created columns to 2 decimal places
new_cols = [
    "MPG", "PPG", "APG", "RPG", "ORPG", "DRPG",
    "PTS_PER_40", "AST_PER_40", "REB_PER_40",
    "ORB_PER_40", "DRB_PER_40",
    "STL_PER_40", "BLK_PER_40",
    "TOV_PER_40", "PF_PER_40", 
    "AST_TO_TOV", "STOCKS_PER_40"
]

nba_player_stats[new_cols] = nba_player_stats[new_cols].round(2)

In [ ]:
nba_player_stats[nba_player_stats['Player'] == 'James Ennis III']

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,AST_PER_40,REB_PER_40,ORB_PER_40,DRB_PER_40,STL_PER_40,BLK_PER_40,TOV_PER_40,PF_PER_40,AST_TO_TOV,STOCKS_PER_40
1811,349,James Ennis III,25,3TM,SF,22,5,329,54,113,...,2.55,5.11,2.55,2.55,1.95,0.61,2.31,3.40,1.11,2.55
1812,247,James Ennis III,26,MEM,SF,64,28,1501,146,321,...,1.71,6.90,1.84,5.06,1.23,0.51,1.57,4.40,1.08,1.73
1813,217,James Ennis III,27,2TM,SF,72,22,1604,182,384,...,1.77,5.59,1.82,3.77,1.17,0.45,1.40,3.34,1.27,1.62
1814,275,James Ennis III,28,2TM,SF,58,27,1230,138,294,...,1.33,5.92,1.95,3.97,1.33,0.75,1.11,4.88,1.21,2.08
1815,214,James Ennis III,29,2TM,SG,69,18,1265,160,359,...,1.99,7.75,2.25,5.50,1.11,0.70,1.83,4.33,1.09,1.80
1816,268,James Ennis III,30,ORL,SF,41,37,986,115,243,...,2.52,6.73,1.74,4.99,1.30,0.28,1.46,3.41,1.72,1.58


Filter to only the college players final season

In [ ]:
college_player_stats_copy = college_player_stats.copy()

In [ ]:
# Remove duplicate player records in same season
college_player_stats_copy = (
    college_player_stats_copy
    .sort_values(["athleteId", "season", "games"])
    .drop_duplicates(
        subset=["athleteId", "season"],
        keep="last"
    )
)

In [ ]:
college_player_stats_copy[college_player_stats_copy['name'] == 'Jalen Johnson']

,season,seasonLabel,teamId,team,conference,athleteId,athleteSourceId,name,position,games,...,TPG,FPG,PTS_PER_40,AST_PER_40,STL_PER_40,BLK_PER_40,TOV_PER_40,FOULS_PER_40,AST_TO_TOV,STOCKS_PER_40
22185,2017,20162017,550,Paul Quinn,NaN,35572,3954888,Jalen Johnson,G,1,...,1.00,4.00,21.94,0.00,2.58,0.00,1.29,5.16,0.00,2.58
20377,2017,20162017,252,Saint Louis,A-10,27997,4066817,Jalen Johnson,F,33,...,1.33,2.67,12.59,1.26,1.16,0.86,2.23,4.45,0.57,2.02
33495,2018,20172018,1092,Presentation College,NaN,49951,4288381,Jalen Johnson,G,3,...,0.67,0.00,10.91,0.91,0.00,0.00,1.82,0.00,0.50,0.00
29511,2018,20172018,292,Tennessee,SEC,25801,4066216,Jalen Johnson,G,13,...,0.15,0.00,9.70,3.03,1.21,1.21,1.21,0.00,2.50,2.42
29023,2018,20172018,252,Saint Louis,A-10,27997,4066817,Jalen Johnson,F,31,...,1.32,2.32,13.82,0.99,0.52,0.99,1.94,3.41,0.51,1.51
42238,2019,20182019,1105,Lenoir-Rhyne,NaN,30042,4402206,Jalen Johnson,G,1,...,3.00,2.00,6.00,0.00,4.00,0.00,6.00,4.00,0.00,4.00
36919,2019,20182019,184,Murray State,OVC,44345,4398349,Jalen Johnson,G,19,...,0.11,0.37,10.00,5.00,0.71,0.00,1.43,5.00,3.50,0.71
38323,2019,20182019,292,Tennessee,SEC,25801,4066216,Jalen Johnson,G,26,...,0.23,0.65,10.33,2.12,0.53,0.26,1.59,4.50,1.33,0.79
48233,2020,20192020,526,Newberry,NaN,2785,4598711,Jalen Johnson,G,1,...,0.00,3.00,0.00,0.00,0.00,0.00,0.00,12.00,NaN,0.00
49961,2020,20192020,823,Anderson (IN),NaN,37979,4600504,Jalen Johnson,ATH,1,...,0.00,1.00,0.00,0.00,0.00,0.00,0.00,3.33,NaN,0.00


In [ ]:
college_final_season = college_player_stats_copy.loc[
    college_player_stats_copy.groupby("athleteId")["season"].idxmax()
].reset_index(drop=True)

In [55]:
college_final_season[college_final_season['name'] == 'Jalen Johnson']

,season,seasonLabel,teamId,team,conference,athleteId,athleteSourceId,name,position,games,...,TPG,FPG,PTS_PER_40,AST_PER_40,STL_PER_40,BLK_PER_40,TOV_PER_40,FOULS_PER_40,AST_TO_TOV,STOCKS_PER_40
2506,2021,20202021,526,Newberry,NaN,2785,4598711,Jalen Johnson,G,1,...,0.00,1.00,5.45,3.64,1.82,0.00,0.00,1.82,NaN,1.82
7847,2025,20242025,597,Eureka,NaN,8730,5185970,Jalen Johnson,G,3,...,1.00,0.33,0.00,0.00,0.00,0.00,5.45,1.82,0.00,0.00
14009,2024,20232024,103,Grambling,SWAC,16334,4594013,Jalen Johnson,F,36,...,0.92,2.25,13.91,1.00,1.11,0.53,1.74,4.27,0.58,1.63
16624,2024,20232024,959,Southwestern Oklahoma State,NaN,19545,5191391,Jalen Johnson,G,1,...,1.00,2.00,23.33,0.00,1.67,0.00,1.67,3.33,0.00,1.67
21586,2022,20212022,164,Mercer,SoCon,25801,4066216,Jalen Johnson,G,32,...,1.44,2.22,17.28,1.74,1.10,1.42,1.87,2.88,0.93,2.52
22134,2022,20212022,315,UIC,Horizon,26412,4702706,Jalen Johnson,G,12,...,0.25,0.33,12.99,3.12,0.52,1.04,1.56,2.08,2.00,1.56
23552,2021,20202021,174,Mississippi State,SEC,27997,4066817,Jalen Johnson,F,24,...,0.58,0.75,12.69,1.02,0.71,0.41,1.42,1.83,0.71,1.12
25105,2022,20212022,1105,Lenoir-Rhyne,NaN,30042,4402206,Jalen Johnson,G,1,...,2.00,2.00,12.86,1.43,4.29,0.00,2.86,2.86,0.50,4.29
26975,2021,20202021,72,Duke,ACC,32426,4701230,Jalen Johnson,F,13,...,2.54,2.23,21.01,4.17,2.16,2.30,4.75,4.17,0.88,4.46
29370,2017,20162017,550,Paul Quinn,NaN,35572,3954888,Jalen Johnson,G,1,...,1.00,4.00,21.94,0.00,2.58,0.00,1.29,5.16,0.00,2.58
